In [26]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

from longeval.spark import get_spark
from pyspark.sql import functions as F

spark = get_spark(cores=8, memory="20g")

data_root = Path("~/shared/longeval/2025/bm25/evaluation").expanduser()
df = spark.read.parquet(data_root.as_posix())
df.printSchema()
df.groupBy("date").agg(F.avg("ndcg_cut_10")).show()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
root
 |-- map: double (nullable = true)
 |-- ndcg: double (nullable = true)
 |-- ndcg_cut_10: double (nullable = true)
 |-- ndcg_rel: double (nullable = true)
 |-- qid: string (nullable = true)
 |-- date: string (nullable = true)



25/05/30 02:38:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+----------------+
|   date|avg(ndcg_cut_10)|
+-------+----------------+
|2022-08|             0.0|
|2022-07|             0.0|
|2022-06|             0.0|
|2023-01|             0.0|
|2022-11|             0.0|
|2022-12|             0.0|
|2023-02|             0.0|
|2022-10|             0.0|
|2022-09|             0.0|
+-------+----------------+



In [ ]:
import pytrec_eval
import tqdm


def _run_df(retrieval, qrels):
    # retrieval should have qid, docid, score (higher score is better)
    run_df = (
        # retrieval has doc{id} whereas qrel just has id
        retrieval.withColumn(
            "docid", F.replace(F.col("docid"), F.lit("doc"), F.lit("")).astype("int")
        )
        .join(
            qrels.select("qid", "docid", "rel"),
            on=["qid", "docid"],
            how="left",
        )
        .fillna(0)
        .orderBy("qid", "docid", "rel")
    )
    return run_df


def _prep_run(run_df):
    qrel = {}
    run = {}
    for row in tqdm.tqdm(run_df.collect()):
        if row.qid not in qrel:
            qrel[row.qid] = {}
        if row.qid not in run:
            run[row.qid] = {}
        qrel[str(row.qid)][str(row.docid)] = row.rel
        run[str(row.qid)][str(row.docid)] = row.score
    return qrel, run


def score_search(
    retrieval,
    qrels,
    scores={"ndcg", "ndcg_rel", "ndcg_cut_10", "map"},
):
    # retrieval should have qid, docid, score (higher score is better)
    run_df = _run_df(retrieval, qrels)
    # now convert to the format required by trec_eval
    qrel, run = _prep_run(run_df)
    evaluator = pytrec_eval.RelevanceEvaluator(qrel, scores)
    evals = evaluator.evaluate(run)

    return get_spark().createDataFrame([{"qid": k, **v} for k, v in evals.items()])

In [ ]:
test_qrel_root = Path("~/scratch/longeval/raw/2025/test-qrels/longeval_web_qrels")
test_qrel = (
    spark.read.csv(
        f"{(test_qrel_root).expanduser().as_posix()}/*/*",
        sep=" ",
        schema="qid STRING, rank INT, docid STRING, rel INT",
    )
    .withColumn(
        "date", F.udf(lambda x: Path(x).parent.name, "string")(F.input_file_name())
    )
    .cache()
)
test_qrel.show(5)
# the error down there doesn't actually mean anything, its a red herring

25/05/30 02:40:23 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /storage/home/hcoda1/8/amiyaguchi3/scratch/longeval/raw/2025/test-qrels/longeval_web_qrels/*/*.
java.io.FileNotFoundException: File /storage/home/hcoda1/8/amiyaguchi3/scratch/longeval/raw/2025/test-qrels/longeval_web_qrels/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$

+-----+----+-------+---+-------+
|  qid|rank|  docid|rel|   date|
+-----+----+-------+---+-------+
|    3|   0|1641821|  2|2023-04|
|32072|   0|1591375|  2|2023-04|
|60646|   0|3472421|  1|2023-04|
|60646|   0|1706779|  0|2023-04|
|60646|   0|3453182|  0|2023-04|
+-----+----+-------+---+-------+
only showing top 5 rows


Now we iterate over all the dates to get scores...

In [36]:
data_root = Path("~/shared/longeval/2025/bm25/retrieval").expanduser()
df = spark.read.parquet(data_root.as_posix())

In [49]:
run_df = _run_df(df, test_qrel).cache()
run_df.show(5)

+---+-----+----+-----------------+-------+---------+---+
|qid|docid|rank|            score|   date|sample_id|rel|
+---+-----+----+-----------------+-------+---------+---+
| 10| 3710|  47|5.735499858856201|2022-08|        1|  0|
| 10| 3710|  44|5.729499816894531|2022-06|        1|  0|
| 10| 5139|  80|5.513199806213379|2022-08|        1|  0|
| 10| 5139|  76|5.508500099182129|2022-06|        1|  0|
| 10| 5272|  56|5.672699928283691|2022-08|        1|  0|
+---+-----+----+-----------------+-------+---------+---+
only showing top 5 rows


In [51]:
test_qrel.groupBy("rel").count().show()
test_qrel.show(n=5)

+---+------+
|rel| count|
+---+------+
|  1| 38823|
|  2| 54282|
|  0|135151|
+---+------+

+-----+----+-------+---+-------+
|  qid|rank|  docid|rel|   date|
+-----+----+-------+---+-------+
|    3|   0|1641821|  2|2023-04|
|32072|   0|1591375|  2|2023-04|
|60646|   0|3472421|  1|2023-04|
|60646|   0|1706779|  0|2023-04|
|60646|   0|3453182|  0|2023-04|
+-----+----+-------+---+-------+
only showing top 5 rows


In [52]:
# all 0 relevancy? tell me it aint so...
run_df.groupBy("rel").count().show()

+---+--------+
|rel|   count|
+---+--------+
|  1|  169376|
|  2|  195216|
|  0|21002516|
+---+--------+



In [60]:
res = []
for date in test_qrel.select("date").distinct().collect():
    date = date.date
    print(f"Processing {date}")
    qrel = test_qrel.filter(F.col("date") == date)
    df_date = df.filter(F.col("date") == date)
    scored = score_search(df_date, qrel).withColumn("date", F.lit(date))
    scored.describe().show()
    res.append(scored)
res

Processing 2023-04


100%|██████████| 1426900/1426900 [00:24<00:00, 59247.32it/s]


{'1038': 0, '6772': 0, '8961': 0, '10529': 0, '12254': 0, '18690': 0, '27513': 0, '133650': 0, '140659': 0, '154284': 0, '155955': 0, '176233': 0, '184833': 0, '186260': 0, '539453': 0, '592324': 0, '650341': 0, '829419': 0, '920127': 0, '931583': 0, '932342': 0, '933520': 0, '933918': 0, '934207': 0, '934857': 0, '935722': 0, '935977': 0, '937500': 0, '937704': 0, '938101': 0, '938747': 0, '943028': 0, '943222': 0, '943357': 0, '944998': 0, '946394': 0, '946689': 0, '948959': 0, '949231': 0, '949455': 0, '950037': 0, '950817': 0, '950962': 0, '951682': 0, '951807': 0, '952383': 0, '952498': 0, '953074': 0, '955320': 0, '955442': 0, '955446': 0, '956232': 0, '956682': 0, '957974': 0, '960499': 0, '963690': 0, '965495': 0, '967332': 0, '967912': 0, '968926': 0, '971980': 0, '972071': 0, '972207': 0, '974141': 0, '974977': 0, '974998': 0, '975214': 0, '975475': 0, '977541': 0, '977715': 0, '979161': 0, '1600331': 0, '1637687': 0, '1645651': 0, '1678000': 0, '1699808': 0, '2051086': 0, '2

25/05/30 03:14:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+-------------------+------------------+-------------------+------------------+------------------+-------+
|summary|                map|              ndcg|        ndcg_cut_10|          ndcg_rel|               qid|   date|
+-------+-------------------+------------------+-------------------+------------------+------------------+-------+
|  count|              14446|             14446|              14446|             14446|             14446|  14446|
|   mean| 0.2807907572822715|0.3814199635018922|0.32305338220840696|0.3629738793332179|40482.479786792195|   NULL|
| stddev|0.34100166480450067|0.3323171032341994|0.35945230474462814|0.3311550280838772|22625.687010744576|   NULL|
|    min|                0.0|               0.0|                0.0|               0.0|              1000|2023-04|
|    max|                1.0|               1.0|                1.0|               1.0|               998|2023-04|
+-------+-------------------+------------------+-------------------+------------

...

100%|██████████| 550095/550095 [00:09<00:00, 60021.68it/s]


{'6772': 0, '8961': 0, '10529': 0, '12254': 0, '18690': 0, '27389': 0, '27513': 0, '133650': 0, '154284': 0, '155955': 0, '176233': 0, '184833': 0, '186260': 0, '539453': 0, '592324': 0, '650341': 0, '920127': 0, '931583': 0, '932342': 0, '933520': 0, '933918': 0, '934207': 0, '934857': 0, '935722': 0, '935977': 0, '937500': 0, '937704': 0, '938101': 0, '938747': 0, '943028': 0, '943222': 0, '943357': 0, '944998': 0, '946394': 0, '948959': 0, '949231': 0, '949455': 0, '950037': 0, '950817': 0, '950962': 0, '951682': 0, '951807': 0, '952383': 0, '952498': 0, '953074': 0, '955320': 0, '955442': 0, '955446': 0, '956232': 0, '956682': 0, '957974': 0, '960499': 0, '963690': 0, '965495': 0, '967332': 0, '967912': 0, '968926': 0, '971980': 0, '972071': 0, '974141': 0, '974998': 0, '975214': 0, '975475': 0, '977541': 0, '977715': 0, '1600331': 0, '1610900': 0, '1637687': 0, '1641568': 0, '1645307': 0, '1678000': 0, '1699808': 0, '2051086': 0, '2159841': 0, '2176329': 0, '2287952': 0, '2433498'

25/05/30 03:16:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------+-------------------+-------------------+-------------------+-------------------+------------------+-------+
|summary|                map|               ndcg|        ndcg_cut_10|           ndcg_rel|               qid|   date|
+-------+-------------------+-------------------+-------------------+-------------------+------------------+-------+
|  count|               5607|               5607|               5607|               5607|              5607|   5607|
|   mean|0.27286106602323856|0.36863334585821983|0.31560713145514646| 0.3555939790266349|33253.375780274655|   NULL|
| stddev|0.34513310144328685|0.33605831370674916| 0.3625815197984668|0.33455499280593365|23375.240884628733|   NULL|
|    min|                0.0|                0.0|                0.0|                0.0|              1000|2023-03|
|    max|                1.0|                1.0|                1.0|                1.0|               998|2023-03|
+-------+-------------------+-------------------+---------------

[DataFrame[map: double, ndcg: double, ndcg_cut_10: double, ndcg_rel: double, qid: string, date: string],
 DataFrame[map: double, ndcg: double, ndcg_cut_10: double, ndcg_rel: double, qid: string, date: string],
 DataFrame[map: double, ndcg: double, ndcg_cut_10: double, ndcg_rel: double, qid: string, date: string],
 DataFrame[map: double, ndcg: double, ndcg_cut_10: double, ndcg_rel: double, qid: string, date: string],
 DataFrame[map: double, ndcg: double, ndcg_cut_10: double, ndcg_rel: double, qid: string, date: string],
 DataFrame[map: double, ndcg: double, ndcg_cut_10: double, ndcg_rel: double, qid: string, date: string]]

In [62]:
# combine all the results
combined = res[0]
for r in res[1:]:
    combined = combined.union(r)
combined.groupBy("date").agg(F.avg("ndcg_cut_10")).show()

+-------+-------------------+
|   date|   avg(ndcg_cut_10)|
+-------+-------------------+
|2023-04|0.32305338220840696|
|2023-08|0.28433060583946274|
|2023-05| 0.3266024299819424|
|2023-07| 0.3192363576872609|
|2023-06| 0.3180535954183221|
|2023-03|0.31560713145514646|
+-------+-------------------+

